In [1]:
import pandas as pd
import numpy as np

import seaborn as sns
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

Mounted at /content/drive


In [2]:
import pandas as pd

# Assuming your file is in Excel format, use read_excel
data = pd.read_csv('/content/drive/MyDrive/new_bangladesh_student/new_Depression3.csv')
data.head()

,University,Department,Depression Level,Age_23-26,Age_27-30,Age_Below 18,Gender_Male,Gender_Prefer not to say,Academic Year_Fourth Year,Academic Year_Second Year,Current CGPA_3.00 - 3.39,Current CGPA_3.40 - 3.79,Current CGPA_3.80 - 4.00,Current CGPA_Below 2.50
0,8,2,3,0,0,0,0,0,1,0,0,0,0,0
1,8,2,2,0,0,0,1,0,0,0,0,0,1,0
2,8,2,4,0,0,0,1,0,0,0,1,0,0,0
3,8,2,2,0,0,0,1,0,0,0,0,1,0,0
4,8,2,2,0,0,0,1,0,0,0,0,1,0,0


In [3]:
import pandas as pd

# Drop the 'Abundance' column from the DataFrame to create the feature matrix X
X = data.drop('Depression Level', axis=1)

# Extract the 'Abundance' column as the target variable y
y = data['Depression Level']

# Print the feature matrix X and target variable y
print(X)
print(y)

      University  Department  Age_23-26  Age_27-30  Age_Below 18  Gender_Male  \
0              8           2          0          0             0            0   
1              8           2          0          0             0            1   
2              8           2          0          0             0            1   
3              8           2          0          0             0            1   
4              8           2          0          0             0            1   
...          ...         ...        ...        ...           ...          ...   
1972           2           0          1          0             0            1   
1973           2          10          1          0             0            0   
1974           2           0          1          0             0            0   
1975           2          10          1          0             0            1   
1976           8           2          1          0             0            1   

      Gender_Prefer not to 

In [4]:
from sklearn.model_selection import train_test_split
# Split the data into train, test, and validation sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Initialize the random forest classifier
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)

# Train the model
rf_classifier.fit(X_train, y_train)


RandomForestClassifier(random_state=42)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns

# Initialize the random forest classifier
rf_classifier = RandomForestClassifier()

# Train the model with noisy data
rf_classifier.fit(X_train, y_train)

# Model testing and evaluation
y_pred = rf_classifier.predict(X_test)

# Calculate accuracy, precision, recall, and F1-score
accuracy = accuracy_score(y_test, y_pred)
precision_weighted = precision_score(y_test, y_pred, average='weighted')
precision_micro = precision_score(y_test, y_pred, average='micro')
precision_macro = precision_score(y_test, y_pred, average='macro')
precision_per_class = precision_score(y_test, y_pred, average=None)

print("Accuracy:", accuracy)
print("Weighted Precision:", precision_weighted)
print("Micro Precision:", precision_micro)
print("Macro Precision:", precision_macro)
print("Precision per class:", precision_per_class)

# Generate and plot classification report
class_report = classification_report(y_test, y_pred)
print("Classification Report:")
print(class_report)

# Calculate ROC curve and AUC
y_pred_probs = rf_classifier.predict_proba(X_test)[:, 1]

# Plot Precision-Recall curve for each class
plt.figure(figsize=(8, 6))
classes = np.unique(y_test)
for class_label in classes:
    y_binary = (y_test == class_label).astype(int)
    class_probs = y_pred_probs  # Use the calculated y_pred_probs
    precision, recall, _ = precision_recall_curve(y_binary, class_probs)
    plt.step(recall, precision, label=f'Class {class_label}')

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve for each class')
plt.legend()
plt.show()

# Plot F1-score curve
plt.figure(figsize=(8, 6))
for class_label in classes:
    y_binary = (y_test == class_label).astype(int)
    precision, recall, _ = precision_recall_curve(y_binary, y_pred_probs)
    f1_values = 2 * (precision * recall) / (precision + recall)
    plt.plot(recall, f1_values, label=f'Class {class_label}')

plt.xlabel('Recall')
plt.ylabel('F1-score')
plt.title('F1-score Curve for each class')
plt.legend()
plt.show()

# Plot Confusion matrix
confusion_mat = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 6))
sns.heatmap(confusion_mat, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

# Calculate ROC curve and AUC for each class
y_pred_probs = rf_classifier.predict_proba(X_test)
fpr = dict()
tpr = dict()
roc_auc = dict()

# Assuming your classes are encoded as integers starting from 0
classes = np.unique(y_test)

for i, class_label in enumerate(classes):
    y_true_binary = (y_test == class_label).astype(int)
    y_score = y_pred_probs[:, i]

    # Calculate ROC curve for the current class
    fpr[class_label], tpr[class_label], _ = roc_curve(y_true_binary, y_score)

    # Calculate AUC for the current class
    roc_auc[class_label] = auc(fpr[class_label], tpr[class_label])

# Plot ROC curves for each class
plt.figure(figsize=(8, 6))
for class_label in classes:
    plt.plot(fpr[class_label], tpr[class_label], label=f'Class {class_label} (AUC = %0.2f)' % roc_auc[class_label])

plt.plot([0, 1], [0, 1], color='r', linestyle='--')
plt.xlim([0, 1])
plt.ylim([0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve for each class')
plt.legend(loc="lower right")
plt.show()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns

# Initialize the logistic regression classifier with a slightly larger regularization (C=10)
logistic_classifier = LogisticRegression(max_iter=1000, C=10, class_weight='balanced')

# Train the model on noisy data
logistic_classifier.fit(X_train, y_train)

# Model testing and evaluation
y_pred = logistic_classifier.predict(X_test)

# Calculate accuracy, precision, recall, and F1-score
accuracy = accuracy_score(y_test, y_pred)
precision_weighted = precision_score(y_test, y_pred, average='weighted')
precision_micro = precision_score(y_test, y_pred, average='micro')
precision_macro = precision_score(y_test, y_pred, average='macro')
precision_per_class = precision_score(y_test, y_pred, average=None)

print("Accuracy:", accuracy)
print("Weighted Precision:", precision_weighted)
print("Micro Precision:", precision_micro)
print("Macro Precision:", precision_macro)
print("Precision per class:", precision_per_class)

# Generate and plot classification report
class_report = classification_report(y_test, y_pred)
print("Classification Report:")
print(class_report)

# Calculate ROC curve and AUC
y_pred_probs = logistic_classifier.predict_proba(X_test)[:, 1]

# Plot Precision-Recall curve for each class
plt.figure(figsize=(8, 6))
classes = np.unique(y_test)
for class_label in classes:
    y_binary = (y_test == class_label).astype(int)
    class_probs = y_pred_probs  # Use the calculated y_pred_probs
    precision, recall, _ = precision_recall_curve(y_binary, class_probs)
    plt.step(recall, precision, label=f'Class {class_label}')

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve for each class')
plt.legend()
plt.show()

# Plot F1-score curve
plt.figure(figsize=(8, 6))
for class_label in classes:
    y_binary = (y_test == class_label).astype(int)
    precision, recall, _ = precision_recall_curve(y_binary, y_pred_probs)
    f1_values = 2 * (precision * recall) / (precision + recall)
    plt.plot(recall, f1_values, label=f'Class {class_label}')

plt.xlabel('Recall')
plt.ylabel('F1-score')
plt.title('F1-score Curve for each class')
plt.legend()
plt.show()

# Plot Confusion matrix
confusion_mat = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 6))
sns.heatmap(confusion_mat, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

# Calculate ROC curve and AUC for each class
y_pred_probs = logistic_classifier.predict_proba(X_test)
fpr = dict()
tpr = dict()
roc_auc = dict()

# Assuming your classes are encoded as integers starting from 0
classes = np.unique(y_test)

for i, class_label in enumerate(classes):
    y_true_binary = (y_test == class_label).astype(int)
    y_score = y_pred_probs[:, i]

    # Calculate ROC curve for the current class
    fpr[class_label], tpr[class_label], _ = roc_curve(y_true_binary, y_score)

    # Calculate AUC for the current class
    roc_auc[class_label] = auc(fpr[class_label], tpr[class_label])

# Plot ROC curves for each class
plt.figure(figsize=(8, 6))
for class_label in classes:
    plt.plot(fpr[class_label], tpr[class_label], label=f'Class {class_label} (AUC = %0.2f)' % roc_auc[class_label])

plt.plot([0, 1], [0, 1], color='r', linestyle='--')
plt.xlim([0, 1])
plt.ylim([0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve for each class')
plt.legend(loc="lower right")
plt.show()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns

# Initialize the Support Vector Classifier with a linear kernel
svc = SVC(kernel='rbf', random_state=42)

# Use CalibratedClassifierCV to enable probability estimates
svc_classifier = CalibratedClassifierCV(svc)


# Train the model on the noisy data
svc_classifier.fit(X_train, y_train)

# Model testing and evaluation
y_pred = svc_classifier.predict(X_test)

# Calculate accuracy, precision, recall, and F1-score
accuracy = accuracy_score(y_test, y_pred)
precision_weighted = precision_score(y_test, y_pred, average='weighted')
precision_micro = precision_score(y_test, y_pred, average='micro')
precision_macro = precision_score(y_test, y_pred, average='macro')
precision_per_class = precision_score(y_test, y_pred, average=None)

print("Accuracy:", accuracy)
print("Weighted Precision:", precision_weighted)
print("Micro Precision:", precision_micro)
print("Macro Precision:", precision_macro)
print("Precision per class:", precision_per_class)

# Generate and plot classification report
class_report = classification_report(y_test, y_pred)
print("Classification Report:")
print(class_report)

# Calculate ROC curve and AUC
y_pred_probs = svc_classifier.predict_proba(X_test)

# Plot Precision-Recall curve for each class
plt.figure(figsize=(8, 6))
classes = np.unique(y_test)
for class_label in classes:
    y_binary = (y_test == class_label).astype(int)
    class_probs = y_pred_probs[:, class_label]
    precision, recall, _ = precision_recall_curve(y_binary, class_probs)
    plt.step(recall, precision, label=f'Class {class_label}')

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve for each class')
plt.legend()
plt.show()

# Plot F1-score curve
plt.figure(figsize=(8, 6))
for class_label in classes:
    y_binary = (y_test == class_label).astype(int)
    precision, recall, _ = precision_recall_curve(y_binary, y_pred_probs[:, class_label])
    f1_values = 2 * (precision * recall) / (precision + recall)
    plt.plot(recall, f1_values, label=f'Class {class_label}')

plt.xlabel('Recall')
plt.ylabel('F1-score')
plt.title('F1-score Curve for each class')
plt.legend()
plt.show()

# Plot Confusion matrix
confusion_mat = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 6))
sns.heatmap(confusion_mat, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

# Calculate ROC curve and AUC for each class
fpr = dict()
tpr = dict()
roc_auc = dict()

for class_label in classes:
    y_true_binary = (y_test == class_label).astype(int)
    y_score = y_pred_probs[:, class_label]

    # Calculate ROC curve for the current class
    fpr[class_label], tpr[class_label], _ = roc_curve(y_true_binary, y_score)

    # Calculate AUC for the current class
    roc_auc[class_label] = auc(fpr[class_label], tpr[class_label])

# Plot ROC curves for each class
plt.figure(figsize=(8, 6))
for class_label in classes:
    plt.plot(fpr[class_label], tpr[class_label], label=f'Class {class_label} (AUC = %0.2f)' % roc_auc[class_label])

plt.plot([0, 1], [0, 1], color='r', linestyle='--')
plt.xlim([0, 1])
plt.ylim([0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve for each class')
plt.legend(loc="lower right")
plt.show()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_curve, accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns


# Initialize the K-Nearest Neighbors Classifier with hyperparameters
knn_classifier = KNeighborsClassifier(n_neighbors=5, weights='uniform', algorithm='auto')

# Train the model using noisy data
knn_classifier.fit(X_train, y_train)

# Model testing and evaluation
y_pred = knn_classifier.predict(X_test)

# Calculate accuracy, precision, recall, and F1-score
accuracy = accuracy_score(y_test, y_pred)
precision_weighted = precision_score(y_test, y_pred, average='weighted')
precision_micro = precision_score(y_test, y_pred, average='micro')
precision_macro = precision_score(y_test, y_pred, average='macro')
precision_per_class = precision_score(y_test, y_pred, average=None)

print("Accuracy:", accuracy)
print("Weighted Precision:", precision_weighted)
print("Micro Precision:", precision_micro)
print("Macro Precision:", precision_macro)
print("Precision per class:", precision_per_class)

# Generate and plot classification report
class_report = classification_report(y_test, y_pred)
print("Classification Report:")
print(class_report)

# Calculate ROC curve and AUC
y_pred_probs = knn_classifier.predict_proba(X_test)

# Plot Precision-Recall curve for each class
plt.figure(figsize=(8, 6))
classes = np.unique(y_test)
for class_label in classes:
    y_binary = (y_test == class_label).astype(int)
    class_probs = y_pred_probs[:, class_label]
    precision, recall, _ = precision_recall_curve(y_binary, class_probs)
    plt.step(recall, precision, label=f'Class {class_label}')

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve for each class')
plt.legend()
plt.show()

# Plot F1-score curve
plt.figure(figsize=(8, 6))
for class_label in classes:
    y_binary = (y_test == class_label).astype(int)
    precision, recall, _ = precision_recall_curve(y_binary, y_pred_probs[:, class_label])
    f1_values = 2 * (precision * recall) / (precision + recall)
    plt.plot(recall, f1_values, label=f'Class {class_label}')

plt.xlabel('Recall')
plt.ylabel('F1-score')
plt.title('F1-score Curve for each class')
plt.legend()
plt.show()

# Plot Confusion matrix
confusion_mat = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 6))
sns.heatmap(confusion_mat, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

# Calculate ROC curve and AUC for each class
fpr = dict()
tpr = dict()
roc_auc = dict()

for class_label in classes:
    y_true_binary = (y_test == class_label).astype(int)
    y_score = y_pred_probs[:, class_label]

    # Calculate ROC curve for the current class
    fpr[class_label], tpr[class_label], _ = roc_curve(y_true_binary, y_score)

    # Calculate AUC for the current class
    roc_auc[class_label] = auc(fpr[class_label], tpr[class_label])

# Plot ROC curves for each class
plt.figure(figsize=(8, 6))
for class_label in classes:
    plt.plot(fpr[class_label], tpr[class_label], label=f'Class {class_label} (AUC = %0.2f)' % roc_auc[class_label])

plt.plot([0, 1], [0, 1], color='r', linestyle='--')
plt.xlim([0, 1])
plt.ylim([0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve for each class')
plt.legend(loc="lower right")
plt.show()


In [ ]:
import pandas as pd
from lightgbm import LGBMClassifier
import re

# Function to clean feature names
def clean_feature_names(df):
    # Replace special characters with underscores
    df.columns = [re.sub(r'\W+', '_', col) for col in df.columns]
    return df

# Assuming X_train is your DataFrame containing features
# Assuming y_train is your target variable
# Assuming X_train and y_train are already defined

# Clean feature names
X_train_clean = clean_feature_names(X_train)

# Initialize the LGBM Classifier
lgbm_classifier = LGBMClassifier()

# Train the model
lgbm_classifier.fit(X_train_clean, y_train)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000325 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 53
[LightGBM] [Info] Number of data points in the train set: 1264, number of used features: 17
[LightGBM] [Info] Start training from score -3.774741
[LightGBM] [Info] Start training from score -3.064499
[LightGBM] [Info] Start training from score -1.558540
[LightGBM] [Info] Start training from score -1.486045
[LightGBM] [Info] Start training from score -1.428304
[LightGBM] [Info] Start training from score -1.370595
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [War

LGBMClassifier()

In [ ]:
import pandas as pd
import numpy as np
from lightgbm import LGBMClassifier
from sklearn.metrics import precision_recall_curve, accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns


# Initialize the LightGBM Classifier with hyperparameters
lgbm_classifier = LGBMClassifier()

# Train the model on the noisy data
lgbm_classifier.fit(X_train, y_train)

# Model testing and evaluation
y_pred = lgbm_classifier.predict(X_test_noisy)
y_pred_probs = lgbm_classifier.predict_proba(X_test)

# Calculate accuracy, precision, recall, and F1-score
accuracy = accuracy_score(y_test, y_pred)
precision_weighted = precision_score(y_test, y_pred, average='weighted')
precision_micro = precision_score(y_test, y_pred, average='micro')
precision_macro = precision_score(y_test, y_pred, average='macro')
precision_per_class = precision_score(y_test, y_pred, average=None)

print("Accuracy:", accuracy)
print("Weighted Precision:", precision_weighted)
print("Micro Precision:", precision_micro)
print("Macro Precision:", precision_macro)
print("Precision per class:", precision_per_class)

# Generate and plot classification report
class_report = classification_report(y_test, y_pred)
print("Classification Report:")
print(class_report)

# Plot Precision-Recall curve for each class
plt.figure(figsize=(8, 6))
classes = np.unique(y_test)
for class_label in classes:
    y_binary = (y_test == class_label).astype(int)
    class_probs = y_pred_probs[:, class_label]
    precision, recall, _ = precision_recall_curve(y_binary, class_probs)
    plt.step(recall, precision, label=f'Class {class_label}')

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve for each class')
plt.legend()
plt.show()

# Plot F1-score curve
plt.figure(figsize=(8, 6))
for class_label in classes:
    y_binary = (y_test == class_label).astype(int)
    precision, recall, _ = precision_recall_curve(y_binary, y_pred_probs[:, class_label])
    f1_values = 2 * (precision * recall) / (precision + recall)
    plt.plot(recall, f1_values, label=f'Class {class_label}')

plt.xlabel('Recall')
plt.ylabel('F1-score')
plt.title('F1-score Curve for each class')
plt.legend()
plt.show()

# Plot Confusion matrix
confusion_mat = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 6))
sns.heatmap(confusion_mat, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

# Calculate ROC curve and AUC for each class
fpr = dict()
tpr = dict()
roc_auc = dict()

for class_label in classes:
    y_true_binary = (y_test == class_label).astype(int)
    y_score = y_pred_probs[:, class_label]

    # Calculate ROC curve for the current class
    fpr[class_label], tpr[class_label], _ = roc_curve(y_true_binary, y_score)

    # Calculate AUC for the current class
    roc_auc[class_label] = auc(fpr[class_label], tpr[class_label])

# Plot ROC curves for each class
plt.figure(figsize=(8, 6))
for class_label in classes:
    plt.plot(fpr[class_label], tpr[class_label], label=f'Class {class_label} (AUC = %0.2f)' % roc_auc[class_label])

plt.plot([0, 1], [0, 1], color='r', linestyle='--')
plt.xlim([0, 1])
plt.ylim([0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve for each class')
plt.legend(loc="lower right")
plt.show()


In [ ]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split

# Initialize the XGBoost classifier
xgb_classifier = XGBClassifier(n_estimators=100, random_state=42, use_label_encoder=False, eval_metric='mlogloss')

# Train the model
xgb_classifier.fit(X_train, y_train)


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [14:34:39] WARNING: /__w/xgboost/xgboost/src/learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import precision_recall_curve, accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns


# Initialize the XGBoost Classifier with hyperparameters
xgb_classifier = XGBClassifier()

# Train the model on the noisy data
xgb_classifier.fit(X_train, y_train)

# Model testing and evaluation
y_pred = xgb_classifier.predict(X_test)
y_pred_probs = xgb_classifier.predict_proba(X_test)

# Calculate accuracy, precision, recall, and F1-score
accuracy = accuracy_score(y_test, y_pred)
precision_weighted = precision_score(y_test, y_pred, average='weighted')
precision_micro = precision_score(y_test, y_pred, average='micro')
precision_macro = precision_score(y_test, y_pred, average='macro')
precision_per_class = precision_score(y_test, y_pred, average=None)

print("Accuracy:", accuracy)
print("Weighted Precision:", precision_weighted)
print("Micro Precision:", precision_micro)
print("Macro Precision:", precision_macro)
print("Precision per class:", precision_per_class)

# Generate and plot classification report
class_report = classification_report(y_test, y_pred)
print("Classification Report:")
print(class_report)

# Plot Precision-Recall curve for each class
plt.figure(figsize=(8, 6))
classes = np.unique(y_test)
for class_label in classes:
    y_binary = (y_test == class_label).astype(int)
    class_probs = y_pred_probs[:, class_label]
    precision, recall, _ = precision_recall_curve(y_binary, class_probs)
    plt.step(recall, precision, label=f'Class {class_label}')

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve for each class')
plt.legend()
plt.show()

# Plot F1-score curve
plt.figure(figsize=(8, 6))
for class_label in classes:
    y_binary = (y_test == class_label).astype(int)
    precision, recall, _ = precision_recall_curve(y_binary, y_pred_probs[:, class_label])
    f1_values = 2 * (precision * recall) / (precision + recall)
    plt.plot(recall, f1_values, label=f'Class {class_label}')

plt.xlabel('Recall')
plt.ylabel('F1-score')
plt.title('F1-score Curve for each class')
plt.legend()
plt.show()

# Plot Confusion matrix
confusion_mat = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 6))
sns.heatmap(confusion_mat, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

# Calculate ROC curve and AUC for each class
fpr = dict()
tpr = dict()
roc_auc = dict()

for class_label in classes:
    y_true_binary = (y_test == class_label).astype(int)
    y_score = y_pred_probs[:, class_label]

    # Calculate ROC curve for the current class
    fpr[class_label], tpr[class_label], _ = roc_curve(y_true_binary, y_score)

    # Calculate AUC for the current class
    roc_auc[class_label] = auc(fpr[class_label], tpr[class_label])

# Plot ROC curves for each class
plt.figure(figsize=(8, 6))
for class_label in classes:
    plt.plot(fpr[class_label], tpr[class_label], label=f'Class {class_label} (AUC = %0.2f)' % roc_auc[class_label])

plt.plot([0, 1], [0, 1], color='r', linestyle='--')
plt.xlim([0, 1])
plt.ylim([0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve for each class')
plt.legend(loc="lower right")
plt.show()


In [ ]:
!pip install deap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 879.5/879.5 kB 7.2 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
from deap import base, creator, tools, algorithms

# Fixed C and solver
C_fixed = 7.3
solver_fixed = 'newton-cg'

# Define evaluation function with added noise
def evaluate_individual(individual):
    # We now use the fixed values for C and solver
    C = C_fixed
    solver = solver_fixed

    try:
        # Define the model with the given hyperparameters
        model = LogisticRegression(C=C, solver=solver, max_iter=1000)

        # Fit the model and evaluate on the validation set
        model.fit(X_train, y_train)
        y_val_pred = model.predict(X_val)


        # Ensure accuracy is not negative and remains in the valid range [0, 1]
        accuracy = max(0, accuracy)

        return accuracy,
    except Exception as e:
        print(f"Error with individual {individual}: {e}")
        return 0.0,

# Create individual and population classes
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

# Register genetic algorithm components
toolbox = base.Toolbox()
toolbox.register("attr_C", lambda: C_fixed)  # Fixed C value
toolbox.register("attr_solver", lambda: solver_fixed)  # Fixed solver value
toolbox.register("individual", tools.initCycle, creator.Individual,
                 (toolbox.attr_C, toolbox.attr_solver), n=1)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

toolbox.register("evaluate", evaluate_individual)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutFlipBit, indpb=0.5)
toolbox.register("select", tools.selTournament, tournsize=3)

# Create the population
population = toolbox.population(n=100)  # Increased population size

# Apply the genetic algorithm
NGEN = 20  # Increased generations
CXPB = 0.7  # Increased crossover probability
MUTPB = 0.3  # Increased mutation probability

for gen in range(NGEN):
    offspring = algorithms.varAnd(population, toolbox, CXPB, MUTPB)
    fits = map(toolbox.evaluate, offspring)

    for fit, ind in zip(fits, offspring):
        ind.fitness.values = fit

    population = toolbox.select(offspring, k=len(population))

# Select the best individual
best_individual = tools.selBest(population, k=1)[0]

# Train and evaluate the final model
final_model = LogisticRegression(C=C_fixed, solver=solver_fixed, max_iter=1000)
final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)

# Calculate and print evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision_weighted = precision_score(y_test, y_pred, average='weighted')
precision_micro = precision_score(y_test, y_pred, average='micro')
precision_macro = precision_score(y_test, y_pred, average='macro')
precision_per_class = precision_score(y_test, y_pred, average=None)

print("Accuracy:", accuracy)
print("Weighted Precision:", precision_weighted)
print("Micro Precision:", precision_micro)
print("Macro Precision:", precision_macro)
print("Precision per class:", precision_per_class)

class_report = classification_report(y_test, y_pred)
print("Classification Report:")
print(class_report)

# Plot Confusion matrix
confusion_mat = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 6))
sns.heatmap(confusion_mat, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

# Calculate ROC curve and AUC for each class
y_pred_probs = final_model.predict_proba(X_test)
fpr = dict()
tpr = dict()
roc_auc = dict()
classes = np.unique(y_test)

for class_label in classes:
    y_true_binary = (y_test == class_label).astype(int)
    y_score = y_pred_probs[:, class_label]

    fpr[class_label], tpr[class_label], _ = roc_curve(y_true_binary, y_score)
    roc_auc[class_label] = auc(fpr[class_label], tpr[class_label])

# Plot ROC curves for each class
plt.figure(figsize=(8, 6))
for class_label in classes:
    plt.plot(fpr[class_label], tpr[class_label], label=f'Class {class_label} (AUC = %0.2f)' % roc_auc[class_label])

plt.plot([0, 1], [0, 1], color='r', linestyle='--')
plt.xlim([0, 1])
plt.ylim([0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve for each class')
plt.legend(loc="lower right")
plt.show()


In [ ]:
import pandas as pd
import numpy as np

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    precision_recall_curve,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    auc
)

import matplotlib.pyplot as plt
import seaborn as sns


# =====================================================
# Data Preprocessing
# =====================================================

# Scale features
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# Encode target labels (for multiclass)
label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)


# Number of classes
num_classes = len(np.unique(y_train_encoded))


# One-hot encoding
y_train_cat = tf.keras.utils.to_categorical(
    y_train_encoded,
    num_classes
)

y_test_cat = tf.keras.utils.to_categorical(
    y_test_encoded,
    num_classes
)


# =====================================================
# ANN Model
# =====================================================

ann_model = Sequential([

    Dense(
        128,
        activation='relu',
        input_shape=(X_train_scaled.shape[1],)
    ),

    Dropout(0.3),

    Dense(
        64,
        activation='relu'
    ),

    Dropout(0.3),

    Dense(
        32,
        activation='relu'
    ),

    Dense(
        num_classes,
        activation='softmax'
    )
])


ann_model.compile(
    optimizer=Adam(
        learning_rate=0.001
    ),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


ann_model.summary()



# =====================================================
# Train ANN (300 Epochs)
# =====================================================

history = ann_model.fit(
    X_train_scaled,
    y_train_cat,
    epochs=300,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)



# =====================================================
# Prediction
# =====================================================

y_pred_probs = ann_model.predict(
    X_test_scaled
)


y_pred_encoded = np.argmax(
    y_pred_probs,
    axis=1
)


y_pred = label_encoder.inverse_transform(
    y_pred_encoded
)



# =====================================================
# Performance Metrics
# =====================================================

accuracy = accuracy_score(
    y_test,
    y_pred
)

precision_weighted = precision_score(
    y_test,
    y_pred,
    average='weighted'
)

precision_micro = precision_score(
    y_test,
    y_pred,
    average='micro'
)

precision_macro = precision_score(
    y_test,
    y_pred,
    average='macro'
)

recall_weighted = recall_score(
    y_test,
    y_pred,
    average='weighted'
)

f1_weighted = f1_score(
    y_test,
    y_pred,
    average='weighted'
)


print("Accuracy:", accuracy)
print("Weighted Precision:", precision_weighted)
print("Weighted Recall:", recall_weighted)
print("Weighted F1-score:", f1_weighted)

print("\nMicro Precision:", precision_micro)
print("Macro Precision:", precision_macro)


print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred
    )
)



# =====================================================
# Precision-Recall Curve
# =====================================================

plt.figure(figsize=(8,6))

classes = np.unique(y_test_encoded)


for class_label in classes:

    y_binary = (
        y_test_encoded == class_label
    ).astype(int)


    precision, recall, _ = precision_recall_curve(
        y_binary,
        y_pred_probs[:, class_label]
    )


    plt.step(
        recall,
        precision,
        label=f'Class {label_encoder.inverse_transform([class_label])[0]}'
    )


plt.xlabel(
    'Recall',
    fontsize=12
)

plt.ylabel(
    'Precision',
    fontsize=12
)

plt.title(
    'Precision-Recall Curve for ANN',
    fontsize=14
)

plt.legend()
plt.grid(alpha=0.3)
plt.show()



# =====================================================
# F1-score Curve
# =====================================================

plt.figure(figsize=(8,6))


for class_label in classes:

    y_binary = (
        y_test_encoded == class_label
    ).astype(int)


    precision, recall, _ = precision_recall_curve(
        y_binary,
        y_pred_probs[:, class_label]
    )


    f1_values = (
        2 *
        (precision * recall)
        /
        (precision + recall + 1e-10)
    )


    plt.plot(
        recall,
        f1_values,
        label=f'Class {label_encoder.inverse_transform([class_label])[0]}'
    )


plt.xlabel(
    'Recall'
)

plt.ylabel(
    'F1-score'
)

plt.title(
    'F1-score Curve for ANN'
)

plt.legend()
plt.grid(alpha=0.3)
plt.show()



# =====================================================
# Confusion Matrix
# =====================================================

confusion_mat = confusion_matrix(
    y_test,
    y_pred
)


plt.figure(
    figsize=(6,6)
)

sns.heatmap(
    confusion_mat,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)


plt.xlabel(
    'Predicted Label'
)

plt.ylabel(
    'True Label'
)

plt.title(
    'Confusion Matrix - ANN'
)

plt.show()



# =====================================================
# ROC Curve and AUC
# =====================================================

fpr = {}
tpr = {}
roc_auc = {}


for class_label in classes:

    y_true_binary = (
        y_test_encoded == class_label
    ).astype(int)


    y_score = y_pred_probs[:, class_label]


    fpr[class_label], tpr[class_label], _ = roc_curve(
        y_true_binary,
        y_score
    )


    roc_auc[class_label] = auc(
        fpr[class_label],
        tpr[class_label]
    )



plt.figure(
    figsize=(8,6)
)


for class_label in classes:

    plt.plot(
        fpr[class_label],
        tpr[class_label],
        label=f'Class {label_encoder.inverse_transform([class_label])[0]} (AUC={roc_auc[class_label]:.2f})'
    )


plt.plot(
    [0,1],
    [0,1],
    'r--'
)


plt.xlabel(
    'False Positive Rate'
)

plt.ylabel(
    'True Positive Rate'
)

plt.title(
    'ROC Curve for ANN'
)

plt.legend(
    loc='lower right'
)

plt.grid(alpha=0.3)

plt.show()

In [ ]:
# =====================================================
# Tabular Transformer Model (300 Epochs)
# =====================================================

import pandas as pd
import numpy as np

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Dense,
    Dropout,
    LayerNormalization,
    MultiHeadAttention,
    Add,
    Flatten
)
from tensorflow.keras.optimizers import Adam

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    roc_curve,
    auc
)

import matplotlib.pyplot as plt
import seaborn as sns



# =====================================================
# Data Preprocessing
# =====================================================

# Feature scaling
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# Encode target labels
label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)


num_classes = len(
    np.unique(y_train_encoded)
)


# One-hot encoding
y_train_cat = tf.keras.utils.to_categorical(
    y_train_encoded,
    num_classes
)

y_test_cat = tf.keras.utils.to_categorical(
    y_test_encoded,
    num_classes
)


# =====================================================
# Transformer Input Reshape
# =====================================================

# Treat each feature as a token
X_train_transformer = np.expand_dims(
    X_train_scaled,
    axis=-1
)

X_test_transformer = np.expand_dims(
    X_test_scaled,
    axis=-1
)


num_features = X_train_transformer.shape[1]



# =====================================================
# Transformer Encoder Block
# =====================================================

def transformer_encoder(
        inputs,
        head_size,
        num_heads,
        ff_dim,
        dropout=0.2):


    # Multi-head attention
    attention_output = MultiHeadAttention(
        num_heads=num_heads,
        key_dim=head_size
    )(
        inputs,
        inputs
    )


    attention_output = Dropout(
        dropout
    )(attention_output)


    # Residual connection
    x = Add()(
        [
            inputs,
            attention_output
        ]
    )


    x = LayerNormalization()(x)


    # Feed-forward network
    ff_output = Dense(
        ff_dim,
        activation="relu"
    )(x)


    ff_output = Dropout(
        dropout
    )(ff_output)


    ff_output = Dense(
        inputs.shape[-1]
    )(ff_output)


    # Residual connection
    x = Add()(
        [
            x,
            ff_output
        ]
    )


    x = LayerNormalization()(x)


    return x



# =====================================================
# Build Transformer Model
# =====================================================

inputs = Input(
    shape=(
        num_features,
        1
    )
)


x = transformer_encoder(
    inputs,
    head_size=32,
    num_heads=4,
    ff_dim=64,
    dropout=0.2
)


x = Flatten()(x)


x = Dense(
    128,
    activation='relu'
)(x)


x = Dropout(
    0.3
)(x)


x = Dense(
    64,
    activation='relu'
)(x)


x = Dropout(
    0.3
)(x)


outputs = Dense(
    num_classes,
    activation='softmax'
)(x)



transformer_model = Model(
    inputs,
    outputs
)



transformer_model.compile(
    optimizer=Adam(
        learning_rate=0.001
    ),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)



transformer_model.summary()



# =====================================================
# Train Transformer (300 Epochs)
# =====================================================

history = transformer_model.fit(
    X_train_transformer,
    y_train_cat,
    epochs=300,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)



# =====================================================
# Prediction
# =====================================================

y_pred_probs = transformer_model.predict(
    X_test_transformer
)


y_pred_encoded = np.argmax(
    y_pred_probs,
    axis=1
)


y_pred = label_encoder.inverse_transform(
    y_pred_encoded
)



# =====================================================
# Performance Metrics
# =====================================================

accuracy = accuracy_score(
    y_test,
    y_pred
)


precision_weighted = precision_score(
    y_test,
    y_pred,
    average='weighted'
)


recall_weighted = recall_score(
    y_test,
    y_pred,
    average='weighted'
)


f1_weighted = f1_score(
    y_test,
    y_pred,
    average='weighted'
)



print("Accuracy:", accuracy)
print("Weighted Precision:", precision_weighted)
print("Weighted Recall:", recall_weighted)
print("Weighted F1-score:", f1_weighted)


print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred
    )
)



# =====================================================
# Confusion Matrix
# =====================================================

cm = confusion_matrix(
    y_test,
    y_pred
)


plt.figure(
    figsize=(6,6)
)


sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)


plt.xlabel(
    "Predicted Label"
)

plt.ylabel(
    "True Label"
)

plt.title(
    "Confusion Matrix - Transformer"
)

plt.show()



# =====================================================
# ROC-AUC Curve
# =====================================================

classes = np.unique(
    y_test_encoded
)


fpr = {}
tpr = {}
roc_auc = {}


for c in classes:

    y_binary = (
        y_test_encoded == c
    ).astype(int)


    fpr[c], tpr[c], _ = roc_curve(
        y_binary,
        y_pred_probs[:,c]
    )


    roc_auc[c] = auc(
        fpr[c],
        tpr[c]
    )



plt.figure(
    figsize=(8,6)
)


for c in classes:

    plt.plot(
        fpr[c],
        tpr[c],
        label=f"Class {label_encoder.inverse_transform([c])[0]} (AUC={roc_auc[c]:.2f})"
    )


plt.plot(
    [0,1],
    [0,1],
    'r--'
)


plt.xlabel(
    "False Positive Rate"
)

plt.ylabel(
    "True Positive Rate"
)

plt.title(
    "ROC Curve - Transformer"
)

plt.legend()

plt.grid(alpha=0.3)

plt.show()